In [1]:
import concurrent.futures
import multiprocessing
import pathlib
import os

import matplotlib.pyplot
import numpy
import polars
from tqdm.notebook import tqdm

from list_scenarios import list_scenarios_full

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = EVALUATION_DIR / "output_dataset"


os.environ["INPUT_PATH"] = str(INPUT_PATH)

# Dataset Exploration

## Calculate Statistics

In [2]:
def stat_scenario(scenario, prot, l2, stp, l2_mode, data, dns, blk):
    eth_csv = INPUT_PATH / f"{scenario}.wpan.eth.csv.gz"
    app_csv = INPUT_PATH / f"{scenario}.wpan.{'http' if scenario.startswith('https') else 'coap'}.csv.gz"
    eth_lf = polars.scan_csv(eth_csv, separator="\t").with_columns(
        (polars.col("eth.payload").str.len_chars() // 2).alias("eth.payload length"),
        (polars.col("eth.payload").str.head(2).is_in(["17", "18"]) & ("-schc-" in scenario)).alias("is_schc_fragment"),
    ).select("frame.number", "eth.src", "eth.dst", "eth.payload length", "is_schc_fragment")
    lf = eth_lf.join(
        polars.scan_csv(app_csv, separator="\t"),
        on="frame.number",
    )
    if app_csv.name.endswith(".http.csv.gz"):
        lf = lf.with_columns(
            polars.when(
                polars.col(
                    "http2.streamid"
                ).str.split(",").list.eval(
                    polars.element() == "0"
                ).list.all()
            ).then(
                0
            ).otherwise(
                polars.col(
                    "http2.streamid"
                ).str.split(",").list.unique().list.eval(
                    polars.element().str.to_integer()
                ).list.sum()
            ).alias("http2.streamid")
        )
        client_addrs = polars.Series(
            lf.filter(
                polars.col("http2.request_in").is_not_null()
            ).select("eth.dst").unique().collect()
        ).to_list()
        dns_streams = polars.Series(
            lf.filter(
                polars.col("http2.headers.content_type").is_in(
                    ["application/dns+cbor", "application/dns-message"]
                )
            ).select("http2.streamid").unique().collect()
        ).to_list()
        data_streams = polars.Series(
            lf.filter(
                polars.col("http2.headers.content_type").is_in(
                    ["application/cbor", "application/json"]
                )
            ).select("http2.streamid").unique().collect()
        ).to_list()
        lf = lf.with_columns(
            (
                polars.col("eth.src").is_in(client_addrs)
                & polars.col("frame.protocols").str.contains("http2")
            ).alias("is_request"),
            (
                polars.col("eth.dst").is_in(client_addrs)
                & polars.col("frame.protocols").str.contains("http2")
            ).alias("is_response"),
            (
                polars.zeros(lf.select(polars.len()).collect()[0, 0]) > 0
            ).alias("is_block"),
            (
                polars.col("http2.streamid").is_in(dns_streams)
            ).alias("is_dns"),
            (
                polars.col("http2.streamid").is_in(data_streams)
            ).alias("is_data"),
        )
        del data_streams
        del dns_streams
    else:
        lf = lf.with_columns(
            (numpy.right_shift(polars.col("coap.code"), 5) == 0).alias("is_request"),
            (numpy.right_shift(polars.col("coap.code"), 5) != 0).alias("is_response"),
            polars.col("coap.opt.block_size").is_not_null().alias("is_block"),
            polars.col("coap.opt.ctype").is_in(["Unknown Type 553", "Unknown Type 53"]).alias("is_dns"),
            polars.col("coap.opt.ctype").is_in(["application/cbor", "application/json"]).alias("is_data"),
        )
    lf = lf.select("eth.payload length", "is_schc_fragment", "is_request", "is_response", "is_block", "is_dns", "is_data")
    df = polars.DataFrame(
        [
            {
                "scenario": scenario,
                "protocol": prot,
                "l2": "schc" if l2 == "-schc" else "eth",
                "l2_mode": l2_mode.strip("-") if l2_mode else "none",
                "setup": stp,
                "data format": data,
                "DNS format": dns,
                "blocksize": 64 if blk else 1024,
                "frames": lf.select("eth.payload length").count().first().collect().item(),
                "requests": lf.filter(polars.col("is_request")).select("is_request").count().first().collect().item(),
                "responses": lf.filter(polars.col("is_response")).select("is_response").count().first().collect().item(),
                "blockwise frames": lf.filter(polars.col("is_block")).select("is_block").count().first().collect().item(),
                "DNS frames": lf.filter(polars.col("is_dns")).select("is_dns").count().first().collect().item(),
                "data frames": lf.filter(polars.col("is_data")).select("is_data").count().first().collect().item(),
                "SCHC fragments": lf.filter(polars.col("is_schc_fragment")).select("is_schc_fragment").count().first().collect().item(),
                "length μ [bytes]": lf.select("eth.payload length").mean().collect().item(),
                "length σ [bytes]": lf.select("eth.payload length").std().collect().item(),
                "length min [bytes]": lf.select("eth.payload length").min().collect().item(),
                "length 0.05 quantile [bytes]": lf.select("eth.payload length").quantile(.05).collect().item(),
                "length 0.25 quantile [bytes]": lf.select("eth.payload length").quantile(.25).collect().item(),
                "length median [bytes]": lf.select("eth.payload length").median().collect().item(),
                "length 0.75 quantile [bytes]": lf.select("eth.payload length").quantile(.75).collect().item(),
                "length 0.95 quantile [bytes]": lf.select("eth.payload length").quantile(.95).collect().item(),
                "length max [bytes]": lf.select("eth.payload length").max().collect().item(),
            }
        ]
    )
    del eth_lf
    del lf
    return df


res = None

workers = multiprocessing.cpu_count()

if workers > 96:
    workers = 96


scenarios = list(list_scenarios_full())


with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as pool:
    def _stat_scenario(args):
        return stat_scenario(*args)
    for df in tqdm(pool.map(_stat_scenario, scenarios), total=len(scenarios)):
        if res is None:
            res = df
        else:
            new = polars.concat([res, df])
            del res
            del df
            res = new
res = res.sort("scenario")
display(res)
res.write_csv(INPUT_PATH / "stats.csv")

  0%|          | 0/296 [00:00<?, ?it/s]

scenario,protocol,l2,l2_mode,setup,data format,DNS format,blocksize,frames,requests,responses,blockwise frames,DNS frames,data frames,SCHC fragments,length μ [bytes],length σ [bytes],length min [bytes],length 0.05 quantile [bytes],length 0.25 quantile [bytes],length median [bytes],length 0.75 quantile [bytes],length 0.95 quantile [bytes],length max [bytes]
str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,f64,f64,f64,f64,f64,i64
"""coap-d1_cbor_dns_cbor""","""coap""","""eth""","""none""","""d1""","""cbor""","""dns_cbor""",1024,100820,50410,50410,0,50410,50373,0,182.633962,166.930904,58,77.0,107.0,120.0,177.0,551.0,1088
"""coap-d1_cbor_dns_cbor_b64""","""coap""","""eth""","""none""","""d1""","""cbor""","""dns_cbor""",64,356930,178465,178465,339039,87297,227736,0,106.113921,33.901177,57,58.0,79.0,111.0,124.0,158.0,497
"""coap-d1_cbor_dns_message""","""coap""","""eth""","""none""","""d1""","""cbor""","""dns_message""",1024,100818,50409,50409,0,50410,50371,0,200.000327,163.13136,58,81.0,124.0,138.0,204.0,551.0,1088
"""coap-d1_cbor_dns_message_b64""","""coap""","""eth""","""none""","""d1""","""cbor""","""dns_message""",64,407386,203693,203693,398847,131891,227736,0,105.711023,33.992584,57,58.0,72.0,112.0,125.0,157.0,497
"""coap-d1_json_dns_cbor""","""coap""","""eth""","""none""","""d1""","""json""","""dns_cbor""",1024,100820,50410,50410,0,50410,50373,0,195.513688,190.271676,58,77.0,107.0,122.0,189.0,628.0,1089
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""oscore-schc-p2_cbor_dns_messag…","""oscore""","""schc""","""none""","""p2""","""cbor""","""dns_message""",64,453797,250107,203690,0,0,0,92511,76.316463,33.431929,3,20.0,55.0,79.0,97.0,127.0,127
"""oscore-schc-p2_json_dns_cbor""","""oscore""","""schc""","""none""","""p2""","""json""","""dns_cbor""",1024,182009,82640,99369,0,0,0,114726,98.327324,34.575731,3,25.0,79.0,110.0,127.0,127.0,127
"""oscore-schc-p2_json_dns_cbor_b…","""oscore""","""schc""","""none""","""p2""","""json""","""dns_cbor""",64,451159,252255,198904,0,0,0,106352,75.749113,33.713544,3,20.0,53.0,82.0,96.0,127.0,127
